# L15a: Testing a Trading Strategy
Previously, we built synthetic markets and placed paper trades. Today, we ask a simple but important question: How do we know a trading strategy works before we risk real money? Towards this question, we'll test two strategies, the utility basket and betting against beta (BAB), on the real market and on 200 synthetic markets.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe a strategy and the engine that runs it:__ Describe the four choices that make up a strategy. Describe how the engine runs a strategy one trading day at a time, and how a trigger starts a new decision.
> * __Backtest two strategies:__ Run the utility basket and BAB on the real market, as if we had traded them, and compare their wealth with the same investment in `SPY`.
> * __Stress-test two strategies:__ Run both strategies on many synthetic markets, read the range of their outcomes, and test the outcomes with gates.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * A strategy decides which tickers to hold, and how much of each, from the data through each close. Our strategies combine a picking rule, a weighting rule, the market hedge, and the estimates.
> * A backtest runs a strategy on one historical market. The engine values the portfolio at each close, starts a new decision only when a trigger fires, and fills the orders at the next open.
> * A stress test runs the same strategy on many synthetic markets that share the real history. Gates test the typical outcome, how often the strategy fails, and how much it loses on the worst markets.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * How the daily update gives more weight to recent prices.
> * How the market hedge limits the portfolio's market sensitivity.
> * The settings of the six triggers, such as the 8% drawdown.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L15a) lists this lecture's symbols with their meanings and units.

Let's get started!

___

## Examples

__We will work through these examples in lecture:__

> [▶ Backtest the utility basket and betting against beta](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). In this example, we backtest both strategies with the same picking rule on the real market, and compare their wealth with `SPY`. We also compare the preference weights with the market sensitivities, place both strategies on the menu, and run the utility basket with updated estimates.

> [▶ Stress-test the utility basket and betting against beta](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). In this example, we stress-test both strategies on 200 synthetic markets and plot the range of their wealth. We test both strategies with the five gates of the practicum and measure their losses on the worst markets. Then we test the utility basket again with the market hedge on.

__Review on your own:__

> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every ticker with each trading day of 2025 and 2026, starting from the 2016 to 2024 fit. We pick the preference basket once a month and whenever the market turns, and compare it with the basket from the fixed estimates.

___

## Backtesting a Strategy
A __trading strategy__ decides which tickers to hold, and how much of each, from the market data through each day's close. Before we risk real money on a strategy, we'd like to know how the strategy would have done. The first test runs the strategy on the market that happened:

> __Definition (backtest):__ A backtest runs a strategy on past market data, one trading day at a time, as if we had traded it. Each decision uses only the data through that day's close, and its orders fill at the next day's opening prices.

The __trading engine__ runs a strategy one trading day at a time, on real or synthetic prices, and records the wealth, the decisions, and the trades. Each of our strategies is one set of four choices:

* __Picking rule:__ This rule chooses the tickers in the basket.
* __Weighting rule:__ This rule sets how much of the portfolio to hold long or short in each ticker.
* __Market hedge:__ With the hedge on, the engine limits the portfolio's market sensitivity. The limit falls as the market falls.
* __Estimates:__ The engine keeps the 2016 to 2024 fit of the single index model fixed, or updates it at each close with more weight on recent days. A third setting keeps the fit fixed and adds a sector term to each stock's model: the recent growth of its sector that the market does not explain.

The [strategy menu](strategies/CHEME-5660-L15a-Strategy-Menu-Fall-2026.ipynb) describes each rule, and [the trading engine](engine/CHEME-5660-L15a-Trading-Engine-Fall-2026.ipynb) describes how the engine makes each decision. In the examples, we'll compare two strategies that share a picking rule and differ only in the weighting rule. You can choose any picking rule and any two weighting rules on the menu.

### Backtest Algorithm
Each trading day of a test follows the same steps. The engine fills the previous decision's orders at the open. After the close, the engine values the portfolio and checks the __triggers__, the rules that start a new decision, such as a fixed schedule or a large fall in wealth:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Backtest-Day.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three trading days, day t minus 1, day t, and day t plus 1, each drawn as a bar from its open to its close. A dashed gray arc from the close of day t minus 1 to the open of day t reads: fill yesterday's orders. An arrow rises from the close of day t to a box labeled after the close: compute the wealth W of t, update the estimates, and check the six triggers. A gray arrow to the left of the box reads: no trigger, keep the portfolio. An orange arrow labeled a trigger fires rises to an orange box, make a new decision, with three steps. Step 1 picks a basket with the picking rule. Step 2 weights it with the weighting rule within the trading limits, and within the market sensitivity limit if the hedge is on. Step 3 queues the orders. An arrow from step 3 runs down to the open of day t plus 1, labeled orders fill at the open."/>
    </center>
</div>

__One trading day.__ On most days, no trigger fires, and the engine keeps the portfolio. When a trigger fires, the strategy chooses new holdings, and the engine queues the orders. The orders fill at the next open.

__Initialize__: Given a strategy; the opening prices $O_{i}(t)$ and the closing prices $S_{i}(t)$ of the tickers $i=1,2,\dots,N$, and the risk-free growth rate $g_{f}(t)$, on the trading days $t=0,1,\dots,T$ of the test; the starting wealth $W_{0}$; and the time step $\Delta t=1/252$ year (one trading day). Start in cash: $c\gets W_{0}$, and $n_{i}\gets 0$ shares of each ticker.

For each trading day $t=0,1,\dots,T$ __do__:

1. __Fill:__ If orders $\Delta n_{i}$ are queued, trade them at the opening prices: $n_{i}\gets n_{i}+\Delta n_{i}$ and $c\gets c-\sum_{i}\Delta n_{i}\,O_{i}(t)-C(t)$, where $C(t)$ (USD) is the trading cost of the fills.
2. __Value and update:__ If $t>0$, grow positive cash at the risk-free rate, $c\gets c\,e^{g_{f}(t)\Delta t}$, and pay the borrowing fee on any short positions. Value the portfolio at the close: $W(t)\gets c+\sum_{i}n_{i}\,S_{i}(t)$. If the strategy uses updated estimates, __then__ update them with today's growth rates.
3. __Decide:__ If $t=0$ or a trigger fires, __then__ the strategy chooses the target shares $n_{i}^{\star}$ from the data through the close of day $t$, and the engine queues the orders $\Delta n_{i}\gets n_{i}^{\star}-n_{i}$ for the next open. Otherwise, keep the holdings.

__Output__: The wealth $W(t)$ for $t=0,1,\dots,T$, the decisions with the trigger that started each one, and the trades.

The [trading engine](engine/CHEME-5660-L15a-Trading-Engine-Fall-2026.ipynb) lists our six triggers and their settings.

Which strategy ends with more wealth on the real market, and does either one beat the same investment in `SPY`? Let's backtest both strategies:

> __Example:__
>
> [▶ Backtest the utility basket and betting against beta](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). In this example, we backtest both strategies with the same picking rule on the real market, and compare their wealth with `SPY`. We also compare the preference weights with the market sensitivities, place both strategies on the menu, and run the utility basket with updated estimates.

A backtest gives one wealth path for each strategy, on the one market that happened. On a market that went differently, the same strategies could end with more or less wealth. Next, we'll run both strategies on many markets that could happen.

___

## Stress-Testing a Strategy
The problem with a backtest is that it runs on one market that happened. A stress test runs a candidate strategy on many synthetic markets that could happen, and tests the range of outcomes against success criteria, called __gates__.

> __Definition (stress test):__ A stress test runs a strategy on many synthetic markets that share the real history up to a start day and differ after it. The __generator__ draws the future of each market. The generator can be any model that simulates prices.

Our generator is the general factor model that we built previously with the JumpHMM method. On every market, the strategy starts from the same wealth on the start day, and only the synthetic future differs:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Stress-Test.svg" width="860" style="max-width:100%; height:auto;" alt="Thin red lines fan out from one point on a vertical dashed line labeled start day. The point is the starting wealth W 0. Each line is the wealth path on one synthetic market from the generator, over one synthetic year of 252 trading days, and the fan widens over the year. At the right edge, a sideways histogram shows the wealth on the last day over the synthetic markets. A dashed horizontal line marks the starting wealth W 0, and a solid tick above it marks the median. The bars below W 0 are a darker red, labeled failures: end below W 0. The lines that end below W 0 have the same darker red, and the other lines and bars are a light red."/>
    </center>
</div>

__A fan of outcomes.__ Each line is the strategy's wealth on one synthetic market, and the lines spread apart over the year. The histogram at the right shows the wealth on the last day. The darker red paths end below the starting wealth $W_{0}$, and we count these markets as failures.

### Stress-Test Algorithm
__Initialize__: Given a strategy; the real history through the start day $t=0$, which ends with the closing prices $S_{i}(0)$ of the tickers $i=1,2,\dots,N$; a generator $\mathcal{G}$; the starting wealth $W_{0}$; the risk-free growth rate $g_{f}$; the number of synthetic markets $P$; the number of synthetic days $T$, such as $T=252$ for one year; the time step $\Delta t=1/252$ year (one trading day); and the reference year $\tau=1$ year.

For each market $p=1,2,\ldots,P$ __do__:

1. __Draw:__ For each day $t=1,2,\dots,T$ __do__: draw the growth rates of the tickers, $\left(g_{1}(t),\dots,g_{N}(t)\right)\gets\mathcal{G}$, and compound each price from the last real close: $S_{i}(t)\gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$. Each synthetic day has one price, so the opening price is $O_{i}(t)=S_{i}(t)$.
2. __Trade:__ Trade the strategy on market $p$ with the trading engine, to find the strategy's wealth $W(t)$ at each close $t=0,1,\dots,T$. The engine starts with $W(0)\gets W_{0}$ in cash and makes the first decision at the close of day $0$, the last real close. Each day, the engine takes the same steps as in a backtest (fill, value and update, decide), on the synthetic prices $S_{i}(t)$.
3. __Score:__ Compute the growth rates of the wealth, $g_{W}(t)\gets\frac{1}{\Delta t}\ln\left(W(t)/W(t-1)\right)$ for $t=1,2,\dots,T$, and their sample mean $\bar{g}_{W}$ and sample standard deviation $\sigma_{g,W}$. Record the scores of market $p$, such as the Sharpe ratio $\mathrm{SR}_{p}\gets\sqrt{\tau/\Delta t}\,\left(\bar{g}_{W}-g_{f}\right)/\sigma_{g,W}$, the largest drawdown $D_{p}\gets\max_{0\leq t\leq T}\left(1-W(t)/\max_{0\leq s\leq t}W(s)\right)$, the wealth on the last day $W_{p}(T)\gets W(T)$, and the net present value $\operatorname{NPV}_{p}\gets -W_{0}+W(T)\,e^{-g_{f}T\Delta t}$.

__Output__: The $P$ values of each score, such as $\mathrm{SR}_{1},\dots,\mathrm{SR}_{P}$, which the gates test.

### Gates
A gate turns the $P$ values of one score into a pass or a fail. A strategy passes the stress test when it passes every gate. Gates test the outcomes in three ways:

* __The typical outcome:__ A gate can test the median of a score over the $P$ markets, such as the median of the Sharpe ratio $\mathrm{SR}_{1},\dots,\mathrm{SR}_{P}$, the median of the largest drawdown $D_{1},\dots,D_{P}$, or the median of the net present value $\operatorname{NPV}_{1},\dots,\operatorname{NPV}_{P}$. The median shows how the strategy does on a typical market. A gate can also compare the median wealth on the last day with that of a benchmark strategy on the same markets.
* __How often the strategy fails:__ The failure rate is the share of the $P$ markets on which the strategy fails. The simplest failure is losing money, $W_{p}(T)<W_{0}$, which the darker red paths in the figure show. A stricter failure is $\operatorname{NPV}_{p}<0$, which means the strategy ended with less wealth than investing $W_{0}$ at the risk-free rate.
* __How much the strategy loses on the worst markets:__ Let the loss on market $p$ be $L_{p}=W_{0}-W_{p}(T)$. The value at risk $\mathrm{VaR}_{\alpha}$ is the loss that only a share $1-\alpha$ of the $P$ markets exceed, for example 5% of the markets for $\alpha=0.95$. The conditional value at risk $\mathrm{CVaR}_{\alpha}$ is the average loss on the markets with $L_{p}\geq\mathrm{VaR}_{\alpha}$.

Each gate has a limit that the investor chooses, such as a median Sharpe ratio of at least 0.3. Thus, you can make the gates reflect your risk tolerance, and expectations. 

Let's stress-test some strategies with an example.

> __Example:__
>
> [▶ Stress-test the utility basket and betting against beta](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). In this example, we stress-test both strategies on 200 synthetic markets and plot the range of their wealth. We test both strategies with the five gates of the practicum and measure their losses on the worst markets. Then we test the utility basket again with the market hedge on.

The outcomes of a stress test are only as realistic as its generator.

___

## Optional Advanced Material
> [▶ Derive the daily update of the single index model](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb). In this notebook, we derive the running sums that the exponentially weighted least squares update carries from one day to the next. We also show how the 2016 to 2024 fit sets their starting values.

___

## Summary
We compared two strategies that share a picking rule, the utility basket and BAB. We described the engine that runs them, backtested both strategies on the real market, and stress-tested both on 200 synthetic markets.

> __Key Takeaways:__
>
> * __A strategy is four choices, and the engine runs it one trading day at a time:__ Each strategy combined a picking rule, a weighting rule, the market hedge, and the estimates. The engine filled the orders at the open, valued the portfolio at the close, and made each later decision only when a trigger fired.
> * __A backtest gives one historical outcome:__ We ran both strategies on the real market, compared their wealth with `SPY`, and marked the days a trigger fired. Each decision used only the data through that day's close.
> * __A stress test gives a range of outcomes:__ We ran both strategies on 200 synthetic markets that share the real history. Gates tested the typical outcome and how often each strategy ended below the starting wealth.

Both tests use past or synthetic prices. A paper test runs the strategy in a paper account at live market prices.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___